# Same loop, different brain

**Block 2, How It Actually Works**

In Block 1 you saw an agent loop that drove Claude through a small
debugging task. The slides for this block argued that **post-training is
convergent**, every major lab uses some version of `pre-train -> SFT
-> preference learning -> tool-use FT`, which is why models trained by
different companies all *behave like agents*.

This notebook makes that claim tactile. We are going to:

1. Import the **same agent loop** Block 1 walked through (now packaged
   for reuse in `workshop_agent`).
2. Discover which models the workshop's LiteLLM proxy actually fronts.
3. Run the loop against each available model on the **same task**.
4. Compare traces.

Then a bonus experiment on the prompt-side lever (tool descriptions).

## 1. Setup

Same setup as Block 1's notebook. The new line is `from workshop_agent
import ...`, the agent loop, the tool handlers, the schemas, all of it
lives in a tiny package now (`workshop_agent/src/workshop_agent/agent.py`).
We freed up this notebook to focus on what is new.

In [4]:
import os
from pathlib import Path

import litellm

from workshop_agent import TOOLS, Sandbox, make_tool_handlers, run_agent

litellm.api_base = os.environ.get("LITELLM_API_BASE")
litellm.api_key = os.environ.get("LITELLM_API_KEY")

# Reuse Block 1's starter project as the sandbox: no need to copy it.
SANDBOX_ROOT = (Path.cwd() / ".." / ".." / "01-landscape" / "demo" / "starter").resolve()
sandbox = Sandbox(SANDBOX_ROOT)
TOOL_HANDLERS = make_tool_handlers(sandbox)

print("api_base:", litellm.api_base)
print("sandbox: ", sandbox.root)

api_base: https://llmaven-prod-litellm-prod.lemonmoss-19296c81.westus2.azurecontainerapps.io
sandbox:  /Users/anantmittal/Code/ssec/schmidt-sciences-workshop/blocks/01-landscape/demo/starter


## 2. Reset Block 1's starter

Whether or not Block 1's demo ran, we need `converters.py` in its known
buggy state so each model sees the same task.

In [5]:
BUGGY = '''"""Temperature unit conversions.

The Fahrenheit/Celsius converter contains a deliberate bug used by the
Block 1 demo. Do not "fix" it outside the demo flow.
"""


def fahrenheit_to_celsius(f: float) -> float:
    """Convert a temperature in degrees Fahrenheit to degrees Celsius."""
    return (f - 32) * 9 / 5


def celsius_to_fahrenheit(c: float) -> float:
    """Convert a temperature in degrees Celsius to degrees Fahrenheit."""
    return c * 9 / 5 + 32


def kelvin_to_celsius(k: float) -> float:
    """Convert a temperature in Kelvin to degrees Celsius."""
    return k - 273.15
'''

(sandbox.root / "src" / "sci_units" / "converters.py").write_text(BUGGY)
print("reset converters.py to buggy starting state")

reset converters.py to buggy starting state


## 3. What models does the proxy actually front?

We do not know in advance whether the LLM proxy server fronts only
Claude, or also GPT and Gemini, or something else entirely. So we
*ask*, by sending a one-token ping to each candidate alias and
keeping the ones that answer.

This is a useful pattern in its own right: the same notebook will keep
working as the proxy admin adds or removes models.

In [7]:
CANDIDATE_MODELS = [
    "claude-sonnet-4-6",
    "claude-haiku-4-5-20251001",
    # "gpt-5",
    # "gpt-5-mini",
    # "gemini-2.5-pro",
    # "gemini-2.5-flash",
]


def model_is_available(model: str) -> bool:
    try:
        litellm.completion(
            model=model,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=4,
        )
        return True
    except Exception as e:
        print(f"  {model}: unavailable ({type(e).__name__})")
        return False


AVAILABLE = [m for m in CANDIDATE_MODELS if model_is_available(m)]
print()
print("available on this proxy:")
for m in AVAILABLE:
    print(f"  - {m}")

if not AVAILABLE:
    raise RuntimeError(
        "No candidate models reachable. Check LITELLM_API_KEY / LITELLM_API_BASE, "
        "or update CANDIDATE_MODELS to match the aliases your proxy exposes."
    )


available on this proxy:
  - claude-sonnet-4-6
  - claude-haiku-4-5-20251001


## 4. The agent loop, in one line

Block 1's notebook spent five cells building the agent loop from
scratch. We are going to reuse it now via a single import. Read the
signature: `run_agent` is the same function you walked through in
Block 1, parameterized so we can pass a different `model=` each time.

In [8]:
import inspect

print(inspect.signature(run_agent))

(*, model: 'str', messages: 'list[dict[str, Any]]', tool_handlers: 'dict[str, Callable[..., str]]', tools: 'list[dict[str, Any]]' = [{'type': 'function', 'function': {'name': 'read_file', 'description': 'Read a UTF-8 text file from the project. Use this to inspect source code, tests, configs, or AGENTS.md.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'description': "Project-relative path, e.g. 'src/sci_units/converters.py'"}}, 'required': ['path']}}}, {'type': 'function', 'function': {'name': 'write_file', 'description': 'Overwrite a UTF-8 text file with new content. Use after you have decided on the change you want to make. The whole file is replaced.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'description': 'Project-relative path'}, 'content': {'type': 'string', 'description': 'Full new file contents'}}, 'required': ['path', 'content']}}}, {'type': 'function', 'function': {'name': 'run_bash', 'description': 'Run a bash

## 5. A small driver

To compare models cleanly we want **the same starting messages, the same
tools, the same task, fresh state**. This wrapper:

1. Resets `converters.py` (so each model sees the same broken file).
2. Builds a fresh message list (so models cannot peek at each other's traces).
3. Calls `run_agent` with the supplied `model=`.
4. Catches errors so one model failing does not kill the demo.

In [9]:
SYSTEM_PROMPT = """You are a careful coding agent working inside a small Python project.

You have three tools: read_file, write_file, run_bash. All paths are relative
to the project root (the directory containing pyproject.toml).

Your job: investigate problems, propose minimal fixes, and verify by running
the test suite. Keep changes small. Always re-run tests after editing.

When you are done, reply with a short plain-text summary of what you did.
"""

USER_PROMPT = (
    "There is a failing test in this project. "
    "Find it, understand the bug, fix it, and verify with pytest. "
    "Use the smallest possible change."
)


def compare(model: str, *, max_steps: int = 10) -> None:
    print(f"\n{'=' * 70}\nMODEL: {model}\n{'=' * 70}")
    # reset the file so each model sees the same starting state
    (sandbox.root / "src" / "sci_units" / "converters.py").write_text(BUGGY)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]
    try:
        run_agent(
            model=model,
            messages=messages,
            tool_handlers=TOOL_HANDLERS,
            tools=TOOLS,
            max_steps=max_steps,
        )
    except Exception as e:
        print(f"\n=== {model} raised {type(e).__name__}: {e} ===")

## 6. Run it against each available model

Watch each trace. The interesting comparisons:

- Which tool does the model call **first**? (`run_bash pytest`? Or
  `read_file converters.py`?)
- How many turns does it take?
- Does it write the same fix? Does it explain the fix?
- Does any model **fail** in an interesting way? (Failure is data, it
  feeds Block 3.)

In [10]:
for model in AVAILABLE[:3]:  # cap at 3 for time
    compare(model)


MODEL: claude-sonnet-4-6

--- step 1  (finish_reason=tool_calls) ---
[text]
Let me start by exploring the project structure and running the tests to find the failure.
[tool_call] run_bash({"command": "find . -type f -name \"*.py\" | head -50 && echo \"---\" && python -m pytest --tb=short 2>&1 | tail -40"})
[tool_result]
exit_code=0
--- stdout ---
./tests/test_converters.py
./tests/__init__.py
./.venv/bin/activate_this.py
./.venv/lib/python3.12/site-packages/packaging/tags.py
./.venv/lib/python3.12/site-packages/packaging/_musllinux.py
./.venv/lib/python3.12/site-packages/packaging/dependency_groups.py
./.venv/lib/p...

--- step 2  (finish_reason=tool_calls) ---
[tool_call] run_bash({"command": "find . -path ./.venv -prune -o -name \"*.py\" -print | grep -v __pycache__ | xargs grep -l \"fahrenheit\\|celsius\" 2>/dev/null"})
[tool_result]
exit_code=0
--- stdout ---
./tests/test_converters.py
./src/sci_units/__init__.py
./src/sci_units/converters.py
--- stderr ---


--- step 3  (finish_r

## 7. What we just saw

Same loop. Same tools. Same task. Different brains.

If post-training were *not* convergent, if Anthropic, OpenAI, and
Google had taken radically different approaches, we would expect at
least one of these models to refuse, to flail, or to misuse the tools.
Instead, they all (mostly) drove the loop to a passing test, with
small stylistic differences.

> **Convergence is not perfection.** Some models will be faster,
> some will write a more elegant fix, some will explain themselves more
> verbosely. But the *shape* of the behavior is the same, and that
> shape is what `pre-train -> SFT -> RLHF -> tool-use FT` produces
> across labs. The tactile proof is right above this cell.

## 8. Bonus: prompt-side levers are real

The slides claimed: most agent failures are **prompt problems**, not
training problems. Let us prove the contrapositive, *deliberately*
sabotage the prompt and watch behavior degrade.

Below, we copy `TOOLS` but replace the carefully written descriptions
with one-word labels. Same model. Same task. Watch what happens.

(If your model still solves it, great, modern models are robust to a
lot of prompt variation. Then try `VAGUE_TOOLS` with even worse
descriptions: empty strings, misleading text.)

In [ ]:
import copy

VAGUE_TOOLS = copy.deepcopy(TOOLS)
for t in VAGUE_TOOLS:
    t["function"]["description"] = "a tool"
    for prop in t["function"]["parameters"]["properties"].values():
        prop.pop("description", None)


def run_with_tools(model: str, tools_to_use: list) -> None:
    print(f"\n--- {model} with {'verbose' if tools_to_use is TOOLS else 'vague'} tools ---")
    (sandbox.root / "src" / "sci_units" / "converters.py").write_text(BUGGY)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]
    try:
        run_agent(
            model=model,
            messages=messages,
            tool_handlers=TOOL_HANDLERS,
            tools=tools_to_use,
            max_steps=10,
        )
    except Exception as e:
        print(f"=== {model} raised {type(e).__name__}: {e} ===")


# Pick one model and compare the two prompt variants
demo_model = AVAILABLE[0]
run_with_tools(demo_model, TOOLS)
run_with_tools(demo_model, VAGUE_TOOLS)


--- claude-sonnet-4-6 with verbose tools ---


## 9. Bridge to Block 3

You may have noticed something interesting: not every model behaves
identically, and not every prompt variation lands cleanly. Some models
might have looped. Some might have written extra files. Some might have
hallucinated a function name that does not exist in `sci_units`.

That is **data, not bugs**. Each of those failures has a name and a
mitigation, and they map back to the post-training pipeline you just
learned. *(Looping? Probably trained on short trajectories. Hallucinating
a function? Knowledge cutoff or pattern bias. Refusing? RLHF over-tuned.)*

**Block 3** turns this into a teachable taxonomy: common failure modes,
where they come from, and what to do about them.